# 📣 Social Wave: Otimização de Campanhas de Marketing

## 1. Entendimento do Negócio e Contextualização

### 1.1 A empresa

A **Social Wave** é uma marca brasileira (fictícia) de roupas casuais que vende só pela internet. Ela nasceu em São Paulo em 2020 e cresceu apostando em anúncios nas redes sociais e nos buscadores. Hoje quase todas as vendas do site vêm de mídia paga, espalhada em 6 canais:

| Canal | Tipo | Papel na estratégia |
|---|---|---|
| Google Search | Anúncio na busca | Pegar quem já está procurando roupa |
| Google Display | Banners em sites | Lembrar a marca para quem já visitou o site |
| Meta Ads | Facebook e Instagram | Principal canal de vendas |
| TikTok Ads | Vídeos curtos | Alcançar o público jovem |
| YouTube Ads | Vídeos | Divulgar a marca |
| Pinterest Ads | Inspiração de moda | Público que procura ideias de look |

### 1.2 O problema

Em 2024 e 2025 o time de marketing dividiu a verba quase igual entre os canais, ajustando "no feeling". O investimento cresceu, mas o **CPA (custo por aquisição)**, ou seja, quanto a empresa gasta em anúncio para conseguir uma venda, subiu junto. A diretoria desconfia que tem canal recebendo dinheiro demais e canal recebendo de menos.

Para 2026 o orçamento de mídia vai continuar o mesmo de 2025. A pergunta da diretoria é direta: **dá para vender mais com o mesmo dinheiro?**

### 1.3 O teste A/B

Em novembro de 2025 o time testou um criativo novo no Meta Ads. As pessoas foram divididas por sorteio em dois grupos:

- **Grupo A (controle):** viu o criativo atual, uma foto do produto em fundo branco.
- **Grupo B (teste):** viu o criativo novo, um vídeo curto de um cliente usando a peça.

O criativo B parece ter vendido mais, mas o time não sabe se a diferença é real ou se foi sorte.

### 1.4 Perguntas de negócio

1. Quanto investimos e quanto retornou em cada canal (CPA e ROAS)?
2. Em que etapa do funil cada canal perde mais gente (CTR e taxa de conversão)?
3. Como investimento, conversões e CPA mudaram ao longo de 2024 e 2025?
4. A partir de que investimento cada canal começa a saturar, ou seja, cada real a mais traz menos vendas?
5. Como redistribuir o mesmo orçamento entre os canais para ter o máximo de conversões, e quanto o CPA cai com isso?
6. No teste A/B do Meta Ads, o criativo novo converte mais que o atual de verdade ou a diferença foi sorte?

### 1.5 Métricas usadas

| Métrica | O que é | Conta |
|---|---|---|
| Investimento | Quanto foi gasto em anúncios | soma do gasto |
| Impressões | Quantas vezes o anúncio apareceu | soma |
| Cliques | Quantas vezes clicaram no anúncio | soma |
| Conversões | Vendas que vieram do anúncio | soma |
| Receita | Valor das vendas que vieram do anúncio | soma |
| CTR | De cada 100 impressões, quantas viraram clique | cliques ÷ impressões |
| Taxa de conversão | De cada 100 cliques, quantos viraram venda | conversões ÷ cliques |
| CPA | Quanto custa conseguir uma venda | investimento ÷ conversões |
| ROAS | Quantos reais voltam para cada real investido | receita ÷ investimento |

### 1.6 Estrutura do projeto

    social-wave-campaign-optimization/
    ├── social_wave_campaign_optimization.ipynb
    ├── generate_data.py
    ├── channels.csv
    ├── campaigns.csv
    ├── daily_performance.csv
    ├── ab_test.csv
    ├── requirements.txt
    ├── README.md
    └── .gitignore

| Arquivo | Conteúdo |
|---|---|
| `social_wave_campaign_optimization.ipynb` | Notebook com todo o estudo de caso |
| `generate_data.py` | Script que gera as bases fictícias (semente fixa) |
| `channels.csv` | Cadastro dos 6 canais de mídia |
| `campaigns.csv` | Cadastro das campanhas de cada canal |
| `daily_performance.csv` | Resultado diário de cada campanha (investimento, impressões, cliques, conversões e receita) de 2024 e 2025 |
| `ab_test.csv` | Resultado de cada pessoa no teste A/B do Meta Ads (grupo, clicou, comprou) |
| `requirements.txt` | Bibliotecas e versões para reproduzir o projeto |

## 2. Bibliotecas utilizadas

| Biblioteca | Para que serve no projeto |
|---|---|
| `pandas` | Ler os arquivos e trabalhar com as tabelas |
| `numpy` | Contas numéricas |
| `sqlite3` | Rodar consultas SQL em um banco em memória |
| `matplotlib` e `seaborn` | Gráficos |

In [4]:
import sqlite3

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style='white')

## 3. Carregamento dos dados

As bases foram geradas pelo script `generate_data.py` e estão na raiz do projeto. São 4 arquivos:

- `channels.csv` e `campaigns.csv`: cadastros (quem é quem)
- `daily_performance.csv`: o resultado diário das campanhas
- `ab_test.csv`: o resultado de cada pessoa no teste A/B

In [7]:
channels = pd.read_csv('channels.csv')
campaigns = pd.read_csv('campaigns.csv')
performance = pd.read_csv('daily_performance.csv')
ab_test = pd.read_csv('ab_test.csv')

for name,df in [('channels', channels),('campaigns', campaigns), ('performance', performance), ('ab_test', ab_test)]:
    print(f'{name}: {df.shape[0]} linhas e {df.shape[1]} colunas')

channels: 6 linhas e 3 colunas
campaigns: 12 linhas e 5 colunas
performance: 8868 linhas e 7 colunas
ab_test: 40200 linhas e 5 colunas


In [8]:
display(channels.head())
print('\n')
display(campaigns.head())
print('\n')
display(performance.head())
print('\n')
display(ab_test.head())

,channel_id,channel_name,channel_type
0,CH01,Google Search,Busca
1,CH02,Google Display,Display
2,CH03,Meta Ads,Rede social
3,CH04,TikTok Ads,Rede social
4,CH05,YouTube Ads,Vídeo


,campaign_id,channel_id,campaign_name,campaign_type,start_date
0,CP01,CH01,Google Search - Prospecção,Prospecção,2024-01-01
1,CP02,CH01,Google Search - Remarketing,Remarketing,2024-01-01
2,CP03,CH02,Google Display - Prospecção,Prospecção,2024-01-01
3,CP04,CH02,Google Display - Remarketing,Remarketing,2024-01-01
4,CP05,CH03,Meta Ads - Prospecção,Prospecção,2024-01-01


,date,campaign_id,spend,impressions,clicks,conversions,revenue
0,2024-01-01,CP10,184.97,12923,59,5.0,1053.27
1,2024-01-01,CP01,410.72,9174,357,19.0,4130.42
2,2024-01-01,CP02,164.25,3705,140,9.0,1849.71
3,2024-01-01,CP03,472.63,65938,262,5.0,887.84
4,2024-01-01,CP04,180.66,20715,80,7.0,1344.73


,user_id,group,exposure_date,clicked,converted
0,U10582,A,2025-11-10,0,0
1,U26354,A,2025-11-22,0,0
2,U28211,B,2025-11-05,0,0
3,U09435,A,2025-11-05,0,0
4,U36696,B,2025-11-06,0,0
